# Setup NemoClaw and VSS on the Brev instance

This notebook is meant to run **on the Brev instance itself**. It walks through the full Brev-side flow for **NemoClaw / OpenClaw** and the **Video Search and Summarization (VSS)** blueprint: configure or reuse a sandbox, verify the policy and imported skills, pin the tested Docker package versions, prepare the host prerequisites for local NIM-backed profiles, start the host-side VSS Orchestrator MCP server, then deploy and manage VSS from the OpenClaw UI by chatting with the agent.

**Default launchable:**  
[video-search-and-summarization-blueprint](https://brev.nvidia.com/launchable/deploy/now?launchableID=env-2tYIjRXL4eMCbH9Az8mJC5WPAI4)

If you are using that launchable, the VM is expected to already have the main prerequisites in place. If not, use the preflight cells below to confirm what is missing.

**Required prerequisites**

- Run this notebook with **Python 3.11 or newer**. The MCP helper uses Python 3.11 standard-library APIs.
- Make sure the intended VSS checkout is the one resolved by `VSS_REPO_DIR`. By default this notebook uses `~/video-search-and-summarization`; set the `VSS_REPO_DIR` environment variable before launching Jupyter if your checkout lives elsewhere or the host has multiple clones.
- If you are re-onboarding a host that previously ran an older OpenShell gateway, follow the NemoClaw sandbox lifecycle upgrade guidance: <https://docs.nvidia.com/nemoclaw/manage-sandboxes/lifecycle>.

**What this notebook covers**

1. Set required keys and notebook options.
2. Run preflight checks for the local repo, scripts, policy file, MCP helper, and host prerequisites, then pin Docker to the tested package versions.
3. Create or reuse the NemoClaw sandbox, configure the chosen OpenClaw model provider, apply the VSS policy, install VSS skills, and update OpenClaw UI `allowedOrigins`.
4. *(Optional)* Verify the live sandbox state, active policy metadata, OpenClaw webhooks (if enabled), and workspace-installed skills.
5. Prepare the host for local NIM-backed VSS profiles by installing/configuring NGC CLI, logging Docker into `nvcr.io`, and preparing the `services/agent/` environment.
6. Start the host-side VSS Orchestrator MCP server, open the OpenClaw UI and smoke-test it, then deploy and manage VSS from the UI by asking the agent to use the `vss_orchestrator__*` MCP tools. Teardown of a deployed VSS stack is also done from the UI (the agent invokes `docker_down`).
7. *(Optional)* Verify sandbox-to-host reachability through `host.openshell.internal`.

**Security:** prefer `NVIDIA_API_KEY` and `NGC_CLI_API_KEY` from environment variables or Brev secrets. Do **not** commit notebook outputs that contain credentials or live access tokens.


## 1. Settings

Configure the notebook in two parts:

1. **Required settings** — credentials and hardware profile that every deployment needs.
2. **Choose ONE OpenClaw model provider** — pick exactly one of (a) a SOTA cloud model, (b) a locally-hosted OpenAI-compatible model, or (c) a model from build.nvidia.com.

> Run section 1.1, then run **exactly one** of the (a) / (b) / (c) cells. Section 1.3 holds advanced defaults you can usually leave alone.

<span style="color:red"><strong>Important:</strong> set <code>NGC_CLI_API_KEY</code> below, and at least one of <code>NVIDIA_API_KEY</code> / <code>COMPATIBLE_API_KEY</code> via the provider cell you pick. Credentials can also come from the environment or Brev secrets.</span>

### 1.1 Required settings

These apply to every deployment:

- `NGC_CLI_API_KEY` — NVIDIA legacy API key used to pull VSS container artifacts from `nvcr.io`.
- `HARDWARE_PROFILE` — selects the hardware-specific overlay. 


In [ ]:
# ================== Required settings (always set) ==================
NGC_CLI_API_KEY = ""               # NVIDIA Legacy API key — used to pull VSS artifacts from nvcr.io
HARDWARE_PROFILE = "RTXPRO6000BW"  # DGX-SPARK | RTXPRO6000BW | H100 | L40S | OTHER

# ================== Shared OpenClaw model vars (overridden by ONE of (a)/(b)/(c) in 1.2) ==================
NVIDIA_API_KEY = ""
NEMOCLAW_ENDPOINT_URL = ""
NEMOCLAW_MODEL = ""
COMPATIBLE_API_KEY = ""


### 1.2 Choose ONE OpenClaw model provider

Run **exactly one** of the cells below. 

| Option | When to use | Sets |
|---|---|---|
| **(a) SOTA cloud model** | Best agent quality (Claude Opus, GPT-5, …) via any OpenAI-compatible cloud API. | `NEMOCLAW_ENDPOINT_URL`, `NEMOCLAW_MODEL`, `COMPATIBLE_API_KEY` |
| **(b) Local OpenAI-compatible model** | Self-hosted on this box or LAN | `NEMOCLAW_ENDPOINT_URL`, `NEMOCLAW_MODEL`, `COMPATIBLE_API_KEY` |
| **(c) build.nvidia.com NVIDIA-hosted model** | Default path — uses NVIDIA's hosted Nemotron via `integrate.api.nvidia.com`. | `NVIDIA_API_KEY`, optionally `NEMOCLAW_MODEL` |


#### (a) SOTA cloud model — *recommended for best agent quality*



In [ ]:
# (a) SOTA cloud model — fill in these three values, then run.
#

NEMOCLAW_ENDPOINT_URL = ""  # OpenAI-compatible base URL, e.g. "https://api.anthropic.com/v1/"
NEMOCLAW_MODEL        = ""  # Model id at that endpoint, e.g. "claude-opus-4-6"
COMPATIBLE_API_KEY    = ""  # Bearer token (sk-ant-..., sk-proj-..., etc.)

#### (b) Local OpenAI-compatible model — *self-hosted / air-gapped*

Point OpenClaw at a local OpenAI-compatible server you've already started on this host or your LAN. For example, a downloadable NIM from build.nvidia.com


> `COMPATIBLE_API_KEY` is technically required by the installer (it errors if blank), but most local servers ignore the value — any non-empty placeholder works.
> From inside the NemoClaw sandbox, the host is reachable as `host.openshell.internal`. If your server binds only to `127.0.0.1` on the host, rebind it to `0.0.0.0` — the sandbox cannot reach a loopback-only socket via `host.openshell.internal`.

In [ ]:
# (b) Local OpenAI-compatible model — fill in to match your local server, then run.
NEMOCLAW_ENDPOINT_URL = "http://host.openshell.internal:8000/v1"
NEMOCLAW_MODEL        = "nvidia/nemotron-3-super-120b-a12b"  # the id your local server reports
COMPATIBLE_API_KEY    = "EMPTY"                              # most local servers ignore this; must be non-empty


#### (c) build.nvidia.com NVIDIA-hosted model — *default, zero-setup*

Uses NVIDIA's hosted model catalog via `integrate.api.nvidia.com`. Only `NVIDIA_API_KEY` is required; `NEMOCLAW_MODEL` defaults to `nvidia/nemotron-3-super-120b-a12b` inside the installer if left blank.

> Get a key at <https://build.nvidia.com> (format `nvapi-...`). To use a different hosted model, set `NEMOCLAW_MODEL` to its build.nvidia.com id (e.g. `nvidia/llama-3.3-nemotron-super-49b-v1.5`).


In [ ]:
# (c) build.nvidia.com — set NVIDIA_API_KEY; clear the custom-endpoint vars so the installer picks NEMOCLAW_PROVIDER=build.
NVIDIA_API_KEY = ""  # "nvapi-..." from https://build.nvidia.com
NEMOCLAW_MODEL = "qwen/qwen3.5-122b-a10b"  # leave blank to use (nvidia/nemotron-3-super-120b-a12b), or set a build.nvidia.com model id

### 1.3 Advanced settings (defaults — usually leave alone)

Pinned installer ref, agent runtime, Brev secure-link domain (`BREV_LINK_DOMAIN` — autodetected: Skybridge instances use `apps.run.brev.nvidia.com`, legacy ones `brevlab.com`; set explicitly for other domains, e.g. `stg.apps.launchpad.nvidia.com`), OpenClaw webhook plumbing, and VSS LLM/VLM overrides. `EXTERNAL_IP` falls back to `hostname -I` when blank.

In [ ]:
import os
import subprocess
from pathlib import Path


# ================== Brev settings ==================
# Brev secure-link domain for the dashboard URL and the CHAT_UI_URL origin
# baked at onboard. Empty = autodetect (Skybridge -> apps.run.brev.nvidia.com,
# legacy -> brevlab.com). Set explicitly for other domains, e.g.
# "stg.apps.launchpad.nvidia.com".
BREV_LINK_DOMAIN = os.environ.get("BREV_LINK_DOMAIN", "").strip()

# ================== Default Nemoclaw settings ==================
# NemoClaw installer pin. Must be a release that ships the canonical
# `nemoclaw sandbox {policy add, skill install, mcp add, config set, upload}`
# subcommands.
NEMOCLAW_INSTALL_REF = "v0.0.80"
# Sandbox agent runtime: "openclaw" (default) or "hermes".
AGENT_RUNTIME = os.environ.get("AGENT_RUNTIME", "openclaw").strip() or "openclaw"
OPENCLAW_HOOKS_ENABLED = True     # OpenClaw webhooks (/hooks)
OPENCLAW_HOOKS_PATH = "/hooks"

# ================== VSS settings ==================
VSS_LLM_ENDPOINT_URL = ""
VSS_LLM_ENABLE_THINKING = ""
VSS_OPENAI_API_KEY = ""            # blank => fall back to NVIDIA_API_KEY (typical for integrate.api.nvidia.com)

# Remote VLM — set VSS_VLM_ENDPOINT_URL non-empty to force VLM_MODE=remote in generated.env
VSS_VLM_ENDPOINT_URL = ""

LLM_DEVICE_ID = "0"
VLM_DEVICE_ID = "1"
EXTERNAL_IP = ""                    # blank => resolve from `hostname -I`


# ================== Derived (no need to touch) ==================

HOME_DIR = Path.home().resolve()
NVIDIA_API_KEY = (NVIDIA_API_KEY or os.environ.get("NVIDIA_API_KEY", "")).strip()
NGC_CLI_API_KEY = (NGC_CLI_API_KEY or os.environ.get("NGC_CLI_API_KEY", "")).strip()
HARDWARE_PROFILE = (HARDWARE_PROFILE or os.environ.get("HARDWARE_PROFILE", "RTXPRO6000BW")).strip()
EXTERNAL_IP = (EXTERNAL_IP or os.environ.get("EXTERNAL_IP", "")).strip()
if not EXTERNAL_IP:
    try:
        EXTERNAL_IP = subprocess.check_output(["hostname", "-I"], text=True).split()[0]
    except (subprocess.SubprocessError, IndexError):
        EXTERNAL_IP = ""
VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", HOME_DIR / "video-search-and-summarization")).resolve()
NEMOCLAW_REPO_DIR = Path(os.environ.get("NEMOCLAW_REPO_DIR", HOME_DIR / "NemoClaw")).resolve()
DEPLOY_SCRIPTS_DIR = VSS_REPO_DIR / "deploy" / "docker" / "scripts"
NEMOCLAW_PROVIDER = "custom" if NEMOCLAW_ENDPOINT_URL else "build"
POLICY_PATH = VSS_REPO_DIR / "assets" / "vss_nemoclaw_policy.yaml"
SKILLS_DIR = VSS_REPO_DIR / "skills"
# Workspace bootstrap docs are uploaded straight into the sandbox (no plugin).
# Runtime-aware: openclaw uses .openclaw/workspace, hermes uses .hermes/workspace.
WORKSPACE_VARIANT = os.environ.get("OPENCLAW_PLUGIN_VARIANT", "nemoclaw").strip() or "nemoclaw"
_RUNTIME_DIRNAME = ".hermes" if AGENT_RUNTIME == "hermes" else ".openclaw"
WORKSPACE_DIR = (VSS_REPO_DIR / _RUNTIME_DIRNAME / "workspace").resolve()
WORKSPACE_REMOTE_DIR = "/sandbox/.openclaw/workspace"
SANDBOX_CONFIG_PATH = "/sandbox/.openclaw/openclaw.json"
OPENCLAW_HOOKS_TOKEN = subprocess.check_output(["openssl", "rand", "-hex", "32"], text=True).strip() if OPENCLAW_HOOKS_ENABLED else ""
AGENT_DIR = VSS_REPO_DIR / "services" / "agent"
MCP_CONFIG_PATH = DEPLOY_SCRIPTS_DIR / "vss_orchestrator_mcp_config.yml"
ORCHESTRATOR_MCP_HELPER_PATH = DEPLOY_SCRIPTS_DIR / "orchestrator_mcp_helper.py"
ARTIFACT_DIR = VSS_REPO_DIR / ".orchestrator-artifacts"
LOG_PATH = ARTIFACT_DIR / "vss_orchestrator_mcp.log"
UV_BIN_DIR = HOME_DIR / ".local" / "bin" / "uv"
LLM_DEVICE_ID = str(LLM_DEVICE_ID or os.environ.get("LLM_DEVICE_ID", "")).strip()
VLM_DEVICE_ID = str(VLM_DEVICE_ID or os.environ.get("VLM_DEVICE_ID", "")).strip()
MCP_HOST = os.environ.get("VSS_ORCHESTRATOR_MCP_HOST", "0.0.0.0").strip()
MCP_PORT = int(os.environ.get("VSS_ORCHESTRATOR_MCP_PORT", "9988"))
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"
HOST_INTERNAL_ALIAS = "host.openshell.internal"
NEMOCLAW_SANDBOX_NAME = os.environ.get("NEMOCLAW_SANDBOX_NAME", "demo").strip()
NEMOCLAW_INSTALL_REF = os.environ.get("NEMOCLAW_INSTALL_REF", NEMOCLAW_INSTALL_REF).strip()
TEST_PORTS = (3000, 8000, 9988, 30888, 5601, 6006, 9200, 8081, 31000, 9901, 38111, 38112)

print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("NEMOCLAW_INSTALL_REF:", NEMOCLAW_INSTALL_REF)
print("\nHOME_DIR:", HOME_DIR)
print("VSS_REPO_DIR:", VSS_REPO_DIR)
print("NEMOCLAW_REPO_DIR:", NEMOCLAW_REPO_DIR)
print("POLICY_PATH:", POLICY_PATH)
print("SKILLS_DIR:", SKILLS_DIR)
print("WORKSPACE_DIR:", WORKSPACE_DIR)
print("AGENT_DIR:", AGENT_DIR)
print("MCP_CONFIG_PATH:", MCP_CONFIG_PATH)
print("ORCHESTRATOR_MCP_HELPER_PATH:", ORCHESTRATOR_MCP_HELPER_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("LOG_PATH:", LOG_PATH)
print("HARDWARE_PROFILE:", HARDWARE_PROFILE)
print("EXTERNAL_IP:", EXTERNAL_IP or "(unresolved)")
print("AGENT_RUNTIME:", AGENT_RUNTIME)
print("BREV_LINK_DOMAIN:", BREV_LINK_DOMAIN or "(autodetect)")
print("WORKSPACE_VARIANT:", WORKSPACE_VARIANT)
print("LLM_DEVICE_ID:", LLM_DEVICE_ID or "(profile default)")
print("VLM_DEVICE_ID:", VLM_DEVICE_ID or "(profile default)")
print("HOST_INTERNAL_ALIAS:", HOST_INTERNAL_ALIAS)
print("MCP_URL:", MCP_URL)
print("NEMOCLAW_PROVIDER:", NEMOCLAW_PROVIDER)
if NEMOCLAW_ENDPOINT_URL:
    print("NEMOCLAW_ENDPOINT_URL:", NEMOCLAW_ENDPOINT_URL)
    print("NEMOCLAW_MODEL:", NEMOCLAW_MODEL)
    print("COMPATIBLE_API_KEY set:", bool(COMPATIBLE_API_KEY))
if VSS_LLM_ENDPOINT_URL:
    print("VSS_LLM_ENDPOINT_URL:", VSS_LLM_ENDPOINT_URL)
print("VSS_LLM_ENABLE_THINKING:", VSS_LLM_ENABLE_THINKING)
if VSS_VLM_ENDPOINT_URL:
    print("VSS_VLM_ENDPOINT_URL:", VSS_VLM_ENDPOINT_URL)
print("OpenClaw hooks enabled:", OPENCLAW_HOOKS_ENABLED)
if OPENCLAW_HOOKS_ENABLED:
    print("OpenClaw hooks path:", OPENCLAW_HOOKS_PATH)
    print("OpenClaw hooks token set:", bool(OPENCLAW_HOOKS_TOKEN))
print("NGC_CLI_API_KEY set:", bool(NGC_CLI_API_KEY))
print("NVIDIA_API_KEY set:", bool(NVIDIA_API_KEY))


## 2. Preflight

Run the next cell to confirm the expected keys, files, commands are present on the instance.


In [ ]:
import importlib.util
import os
import shutil
from pathlib import Path

RED = "\033[31m"
RESET = "\033[0m"
GREEN = "\033[32m"
YELLOW = "\033[33m"

helper_spec = importlib.util.spec_from_file_location("orchestrator_mcp_helper", ORCHESTRATOR_MCP_HELPER_PATH)
if helper_spec is None or helper_spec.loader is None:
    raise ImportError(f"Could not load MCP helper from {ORCHESTRATOR_MCP_HELPER_PATH}")
orchestrator_mcp_helper = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(orchestrator_mcp_helper)

resolve_openshell_gateway_container = orchestrator_mcp_helper.resolve_openshell_gateway_container
detect_brev_link_domain = orchestrator_mcp_helper.detect_brev_link_domain

if not shutil.which("uv") and UV_BIN_DIR.exists():
    os.environ["PATH"] = f"{UV_BIN_DIR.parent}:{os.environ.get('PATH', '')}"

required_checks = {
    "vss_nemoclaw_policy.yaml": POLICY_PATH.is_file(),
    "skills/": SKILLS_DIR.is_dir(),
    "workspace bootstrap dir": WORKSPACE_DIR.is_dir(),
    "services/agent/": AGENT_DIR.is_dir(),
    "vss_orchestrator_mcp_config.yml": MCP_CONFIG_PATH.is_file(),
    "orchestrator_mcp_helper.py": ORCHESTRATOR_MCP_HELPER_PATH.is_file(),
    "docker": shutil.which("docker") is not None,
    "python3": shutil.which("python3") is not None,
    "curl": shutil.which("curl") is not None,
    "uv": shutil.which("uv") is not None,
    "NGC_CLI_API_KEY set": bool(NGC_CLI_API_KEY),
}

if OPENCLAW_HOOKS_ENABLED:
    required_checks["OPENCLAW_HOOKS_TOKEN"] = bool(OPENCLAW_HOOKS_TOKEN)
    required_checks["OPENCLAW_HOOKS_PATH"] = OPENCLAW_HOOKS_PATH

for label, ok in required_checks.items():
    status = "OK " if ok else "NO "
    color = GREEN if ok else RED
    print(f"{color}{status}{RESET} {label}")


### 2.1 Pin Docker version

Pin Docker CE + plugins + containerd.io to a known-good combination (CE **29.4.3**, buildx **0.33.0**, compose **5.1.3**, containerd **2.2.3**) **before** section 3 brings up the OpenClaw sandbox — a docker-ce downgrade restarts dockerd and would disrupt live sandbox containers if it ran later in the notebook. `apt-mark hold` prevents drift back to newer versions before section 6 runs.

The cell first reads the installed Docker Engine version: if it already falls in the tested range **[28.3.3, 29.5.0)** the version downgrade is **skipped** — re-pinning to an exact version the platform's apt repo may not carry (e.g. DGX Spark / DGX-OS on arm64) would fail with *version not found* for no benefit. The packages are still `apt-mark hold`-ed at their current versions so the box can't drift past the tested range mid-run. Safe to re-run.

In [ ]:
%%bash
# Pin Docker CE + plugins + containerd.io to a known-good combination, but
# ONLY when the host's Docker is outside the tested range. Some Brev
# launchables ship a newer Docker than the VSS deploy profiles are tested
# against; pin explicitly so compose/buildx incompatibilities don't surface
# mid-deployment.
#
# When the installed Docker already falls in [28.3.3, 29.5.0) the version
# downgrade is skipped: re-pinning to an exact epoch-versioned package that
# the platform's apt repo may not carry (e.g. DGX Spark / DGX-OS on arm64)
# fails with "version not found" for no benefit. The in-range packages are
# still held so the box can't drift past the tested range mid-notebook.
# Idempotent -- safe to re-run.

set -euo pipefail

# Tested Docker Engine range -- keep in sync with the VSS launchable prereq check.
MIN_DOCKER_VERSION="28.3.3"
MAX_DOCKER_VERSION="29.5.0"

# Packages frozen with `apt-mark hold` so unattended-upgrades / later
# `apt-get install` calls can't drift the box before the notebook finishes.
HOLD_PKGS="docker-ce docker-ce-cli docker-buildx-plugin docker-compose-plugin containerd.io"

version_ge() { [ "$(printf '%s\n%s\n' "$2" "$1" | sort -V | head -n1)" = "$2" ]; }
version_lt() { [ "$1" != "$2" ] && [ "$(printf '%s\n%s\n' "$1" "$2" | sort -V | head -n1)" = "$1" ]; }

DOCKER_VERSION="$(docker version --format '{{.Server.Version}}' 2>/dev/null || true)"
if [ -n "$DOCKER_VERSION" ] \
   && version_ge "$DOCKER_VERSION" "$MIN_DOCKER_VERSION" \
   && version_lt "$DOCKER_VERSION" "$MAX_DOCKER_VERSION"; then
  echo "Docker $DOCKER_VERSION is within the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); skipping the Docker version pin."
  # No downgrade needed, but still hold the in-range packages at their
  # current versions so unattended-upgrades / later apt-get calls can't drift
  # the box past the tested range for the remainder of the notebook.
  sudo apt-mark hold $HOLD_PKGS
  exit 0
fi

if [ -n "$DOCKER_VERSION" ]; then
  echo "Docker $DOCKER_VERSION is outside the tested range [$MIN_DOCKER_VERSION, $MAX_DOCKER_VERSION); pinning to known-good versions."
else
  echo "Could not read the installed Docker version; pinning to known-good versions."
fi

# Read distro info from /etc/os-release (always present on Ubuntu; minimal
# images don't ship `lsb_release`).
. /etc/os-release
DISTRO="${VERSION_ID}"
CODENAME="${UBUNTU_CODENAME:-${VERSION_CODENAME}}"

# Versions hard-coded to what shipped alongside docker-ce 29.4.3 on the
# Docker apt repo (verified against download.docker.com + upstream GitHub
# release timestamps). When bumping DOCKER_CE_VER, bump these four together.
DOCKER_CE_VER="5:29.4.3-1~ubuntu.${DISTRO}~${CODENAME}"
BUILDX_VER="0.33.0-1~ubuntu.${DISTRO}~${CODENAME}"
COMPOSE_VER="5.1.3-1~ubuntu.${DISTRO}~${CODENAME}"
CONTAINERD_VER="2.2.3-1~ubuntu.${DISTRO}~${CODENAME}"

# Refresh the APT cache first -- without this, the specific epoch-versioned
# package may not be in the local index and the install would fail with
# version-not-found before any pinning takes effect.
sudo apt-get update -qq

sudo DEBIAN_FRONTEND=noninteractive apt-get install -y \
  --allow-downgrades \
  -o Dpkg::Options::=--force-confdef \
  -o Dpkg::Options::=--force-confold \
  docker-ce="$DOCKER_CE_VER" \
  docker-ce-cli="$DOCKER_CE_VER" \
  docker-buildx-plugin="$BUILDX_VER" \
  docker-compose-plugin="$COMPOSE_VER" \
  containerd.io="$CONTAINERD_VER"

# Hold so unattended-upgrades / later `apt-get install` calls don't drift
# the box back to newer versions before the rest of the notebook runs.
sudo apt-mark hold $HOLD_PKGS

## 3. Install and Configure NemoClaw (OpenClaw) for VSS skills

Sections 3.1–3.7 install and configure the sandbox using canonical NemoClaw / OpenShell / OpenClaw commands. Run the cells in order — each step is idempotent and safe to re-run.

| Step | What it does |
|---|---|
| 3.1 | Install NemoClaw (pinned `NEMOCLAW_INSTALL_REF`) and create the sandbox |
| 3.2 | Apply the VSS sandbox policy |
| 3.3 | Install the VSS skills |
| 3.4 | Upload the workspace bootstrap docs |
| 3.5 | Register the VSS Orchestrator MCP |
| 3.6 | Configure optional webhooks |
| 3.7 | Verify the dashboard and print the OpenClaw UI URL |


### 3.1 Install NemoClaw and create the sandbox

Exports the NemoClaw configuration (including `CHAT_UI_URL`, the dashboard origin for UI access) and installs NemoClaw at the pinned `NEMOCLAW_INSTALL_REF`. The non-interactive installer also creates and onboards the sandbox. Takes a few minutes; output streams live.


In [ ]:
import json
import os
import re
import shlex
import shutil
import time
from pathlib import Path

# Fail fast if no model provider is configured (section 1.2).
if NEMOCLAW_PROVIDER == "build" and not NVIDIA_API_KEY:
    raise RuntimeError(
        "No OpenClaw provider configured. Pick one and fill in its values:\n"
        "  - option (a) SOTA cloud / (b) local: set NEMOCLAW_ENDPOINT_URL, NEMOCLAW_MODEL, and COMPATIBLE_API_KEY in the (a) or (b) cell\n"
        "  - option (c) build.nvidia.com: set NVIDIA_API_KEY in the (c) cell"
    )
if NEMOCLAW_PROVIDER == "custom" and not COMPATIBLE_API_KEY:
    raise RuntimeError(
        "COMPATIBLE_API_KEY is required for the custom OpenAI-compatible provider (options a/b). "
        "Set it in the (a) or (b) cell (any non-empty placeholder works for most local servers)."
    )
if NEMOCLAW_PROVIDER == "custom" and not NEMOCLAW_MODEL:
    raise RuntimeError(
        "NEMOCLAW_MODEL is required when NEMOCLAW_ENDPOINT_URL is set. "
        "If you ran more than one of (a)/(b)/(c), re-run just the (a) or (b) cell you want to use."
    )

DASHBOARD_PORT = int(os.environ.get("NEMOCLAW_DASHBOARD_PORT", "18789"))


def _brev_env_id():
    val = os.environ.get("BREV_ENV_ID", "").strip()
    if val:
        return val
    try:
        for line in Path("/etc/environment").read_text().splitlines():
            if line.startswith("BREV_ENV_ID="):
                return line.split("=", 1)[1].strip().strip('"').strip("'")
    except OSError:
        pass
    return ""


_env_id = _brev_env_id()

# NemoClaw reads its configuration from environment variables. `env` is also
# reused by the orchestrator MCP cell in section 6.1.
env = os.environ.copy()
env["VSS_REPO_DIR"] = str(VSS_REPO_DIR)
env["NEMOCLAW_SANDBOX_NAME"] = NEMOCLAW_SANDBOX_NAME
env["NEMOCLAW_INSTALL_REF"] = NEMOCLAW_INSTALL_REF
env["NEMOCLAW_PROVIDER"] = NEMOCLAW_PROVIDER
env["NEMOCLAW_NON_INTERACTIVE"] = "1"
env["NEMOCLAW_ACCEPT_THIRD_PARTY_SOFTWARE"] = "1"
env["NVIDIA_API_KEY"] = NVIDIA_API_KEY
env["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY
if NEMOCLAW_MODEL:
    env["NEMOCLAW_MODEL"] = NEMOCLAW_MODEL
if NEMOCLAW_PROVIDER == "custom":
    env["NEMOCLAW_ENDPOINT_URL"] = NEMOCLAW_ENDPOINT_URL
    env["COMPATIBLE_API_KEY"] = COMPATIBLE_API_KEY
    print(f"[{HARDWARE_PROFILE}] Custom OpenAI-compatible provider: {NEMOCLAW_ENDPOINT_URL} model={NEMOCLAW_MODEL} key set={bool(COMPATIBLE_API_KEY)}")
# CHAT_UI_URL must be correct at onboard time: the sandbox derives the UI
# allowed origins and the 0.0.0.0 forward bind from it (gateway.* is read-only later).
BREV_LINK_DOMAIN_RESOLVED = BREV_LINK_DOMAIN or detect_brev_link_domain()
env["BREV_LINK_DOMAIN"] = BREV_LINK_DOMAIN_RESOLVED
if _env_id:
    env["CHAT_UI_URL"] = f"https://{DASHBOARD_PORT}-{_env_id}.{BREV_LINK_DOMAIN_RESOLVED}"
    print("CHAT_UI_URL:", env["CHAT_UI_URL"])
os.environ.update(env)  # export for the ! commands below


def _version_of(text):
    match = re.search(r"v?(\d+\.\d+\.\d+)", text or "")
    return match.group(1) if match else None


install_ref = NEMOCLAW_INSTALL_REF.strip()
if not install_ref:
    raise RuntimeError("NEMOCLAW_INSTALL_REF must not be empty.")
install_url = f"https://raw.githubusercontent.com/NVIDIA/NemoClaw/{install_ref}/install.sh"


In [ ]:
installed = None
if shutil.which("nemoclaw"):
    _ver = !nemoclaw --version 2>&1
    installed = _version_of(" ".join(_ver))

if installed and installed == _version_of(install_ref):
    print(f"NemoClaw {installed} already installed, skipping installer.", flush=True)
else:
    print(f"Installing NemoClaw {install_ref} (takes a few minutes)...", flush=True)
    !cd ~ && bash -o pipefail -c "curl -fsSL {install_url} | bash"
    assert _exit_code == 0, "NemoClaw install failed"

!nemoclaw --version
assert _exit_code == 0, "nemoclaw is not runnable after install"

# The non-interactive installer runs `nemoclaw onboard` itself, so the sandbox
# normally exists already. This guard covers pre-baked images where a matching
# nemoclaw was preinstalled (installer skipped above) but never onboarded.
!openshell sandbox get {NEMOCLAW_SANDBOX_NAME} >/dev/null 2>&1
if _exit_code != 0:
    print(f"No sandbox {NEMOCLAW_SANDBOX_NAME!r} yet — onboarding (agent={AGENT_RUNTIME}, takes several minutes)...", flush=True)
    !cd ~ && nemoclaw onboard --non-interactive --agent {AGENT_RUNTIME}
    assert _exit_code == 0, "nemoclaw onboard failed"
print(f"Sandbox {NEMOCLAW_SANDBOX_NAME!r} ready.", flush=True)


### 3.2 Apply the VSS sandbox policy

Merges `assets/vss_nemoclaw_policy.yaml` into the base OpenShell policy.


In [ ]:
if not POLICY_PATH.is_file():
    raise FileNotFoundError(f"Missing policy file: {POLICY_PATH}")
!nemoclaw sandbox policy add {NEMOCLAW_SANDBOX_NAME} --from-file {POLICY_PATH} --yes
assert _exit_code == 0, "policy add failed"


### 3.3 Install the VSS skills

Installs each `skills/<name>/` directory containing a `SKILL.md`.


In [ ]:
if not SKILLS_DIR.is_dir():
    raise FileNotFoundError(f"Missing skills dir: {SKILLS_DIR}")
skill_dirs = sorted({sp.parent for sp in SKILLS_DIR.glob("*/SKILL.md")})
if not skill_dirs:
    raise RuntimeError(f"No SKILL.md directories found under {SKILLS_DIR}")
for skill_dir in skill_dirs:
    !nemoclaw sandbox skill install {NEMOCLAW_SANDBOX_NAME} {skill_dir}
    assert _exit_code == 0, f"skill install failed: {skill_dir.name}"
print(f"Installed {len(skill_dirs)} VSS skills.", flush=True)


### 3.4 Upload the workspace bootstrap docs

Pushes the base `.md` docs, then the `_<variant>` overlay (overlay last, so it wins).


In [ ]:
if not WORKSPACE_DIR.is_dir():
    print(f"No workspace dir at {WORKSPACE_DIR}; skipping workspace upload.", flush=True)
else:
    # Self-heal: an earlier upload form used a file-path dest, which the
    # directory-semantics transport turned into <name>.md/<name>.md nesting on
    # fresh sandboxes. Remove any directory-shaped *.md leftovers first.
    !nemoclaw sandbox exec {NEMOCLAW_SANDBOX_NAME} -- sh -c "find /sandbox/.openclaw/workspace -mindepth 1 -maxdepth 1 -type d -name '*.md' -exec rm -rf '{{}}' ';'"
    docs = sorted(WORKSPACE_DIR.glob("*.md"))
    overlay_dir = WORKSPACE_DIR / f"_{WORKSPACE_VARIANT}"
    if overlay_dir.is_dir():
        docs += sorted(overlay_dir.glob("*.md"))
    for doc in docs:
        # dest is a DIRECTORY: the OpenShell transport does mkdir + tar-extract
        # into it; a file path here collides with an existing file of that name.
        !nemoclaw sandbox upload {NEMOCLAW_SANDBOX_NAME} {doc} {WORKSPACE_REMOTE_DIR}/ >/dev/null
        assert _exit_code == 0, f"upload failed: {doc.name}"
    print(f"Uploaded {len(docs)} workspace docs to {WORKSPACE_REMOTE_DIR}.", flush=True)


### 3.5 Register the VSS Orchestrator MCP

Registers the host-side VSS Orchestrator MCP. Skipped if already registered.


In [ ]:
!nemoclaw sandbox mcp {NEMOCLAW_SANDBOX_NAME} status vss_orchestrator >/dev/null 2>&1
if _exit_code == 0:
    print("MCP server 'vss_orchestrator' already registered.", flush=True)
else:
    !nemoclaw sandbox mcp {NEMOCLAW_SANDBOX_NAME} add vss_orchestrator --url {MCP_URL}
    assert _exit_code == 0, "mcp add failed"


### 3.6 Configure optional webhooks

Writes the OpenClaw webhook keys (`hooks.*`); the gateway restarts once, after the final write. Skipped when `OPENCLAW_HOOKS_ENABLED` is off.


In [ ]:
# Only the webhook keys need config set:
# - gateway.* is off-limits (auth tokens) — UI allowedOrigins come from
#   CHAT_UI_URL baked at onboard (section 3.1);
# - agents.defaults.workspace already defaults to ~/.openclaw/workspace,
#   which is /sandbox/.openclaw/workspace under the sandbox HOME.
config_sets = []
if OPENCLAW_HOOKS_ENABLED:
    config_sets.append(("hooks.enabled", "true"))
    config_sets.append(("hooks.path", OPENCLAW_HOOKS_PATH or "/hooks"))
    if OPENCLAW_HOOKS_TOKEN:
        config_sets.append(("hooks.token", OPENCLAW_HOOKS_TOKEN))

if not config_sets:
    print("No sandbox config changes needed (webhooks disabled).", flush=True)
for _i, (_key, _value) in enumerate(config_sets):
    _quoted = shlex.quote(_value)  # JSON values contain spaces/quotes
    _restart = "--restart" if _i == len(config_sets) - 1 else ""
    !nemoclaw sandbox config set {NEMOCLAW_SANDBOX_NAME} --key {_key} --value {_quoted} --config-accept-new-path {_restart}
    assert _exit_code == 0, f"config set failed: {_key}"
if config_sets:
    print("Sandbox config applied; gateway restarted.", flush=True)


### 3.7 Verify the dashboard + OpenClaw UI URL

Checks the dashboard health endpoint, repairs the gateway and port-forward with `nemoclaw sandbox recover` if needed, and prints the OpenClaw UI URL.


In [ ]:
# The dashboard forward is OWNED BY ONBOARD: with a remote CHAT_UI_URL
# (section 3.1) it binds 0.0.0.0 so Brev's ingress can reach it (NemoClaw
# dashboard contract #3259; NEMOCLAW_DASHBOARD_BIND=0.0.0.0 is the explicit
# override). Do NOT run `openshell forward start <port>` manually here — a
# plain port target rebinds the forward to loopback and breaks remote access.
# Verify health; if down, `sandbox recover` repairs gateway + forwards.
!curl -fsS -o /dev/null -m 5 --retry 3 --retry-delay 2 --retry-connrefused http://127.0.0.1:{DASHBOARD_PORT}/health
if _exit_code != 0:
    print("Dashboard not serving on localhost — running canonical recovery...", flush=True)
    !nemoclaw sandbox recover {NEMOCLAW_SANDBOX_NAME}
    !curl -fsS -o /dev/null -m 5 --retry 5 --retry-delay 2 --retry-connrefused http://127.0.0.1:{DASHBOARD_PORT}/health
    assert _exit_code == 0, "dashboard still not serving after recover — check `nemoclaw sandbox logs`"
print(f"Dashboard healthy on http://127.0.0.1:{DASHBOARD_PORT}/", flush=True)

_tok_lines = !nemoclaw sandbox gateway token {NEMOCLAW_SANDBOX_NAME} --quiet 2>/dev/null
_token = next((tl.strip() for tl in reversed(_tok_lines) if tl.strip()), "")
_base = (f"https://{DASHBOARD_PORT}-{_env_id}.{BREV_LINK_DOMAIN_RESOLVED}"
         if _env_id else f"http://127.0.0.1:{DASHBOARD_PORT}")
_ui = f"{_base}/chat?session=main#token={_token}" if _token else f"{_base}/chat?session=main"
print("\n" + "=" * 100)
print(f"OpenClaw UI: {_ui}")
print("=" * 100, flush=True)
print("Done.", flush=True)


## 4. [OPTIONAL] Verify sandbox, policy, skills, and optional webhooks

The next cell checks:

- whether the sandbox exists,
- the current active sandbox policy metadata,
- the expected local policy path and version,
- whether OpenClaw webhooks accept a local test request when enabled,
- the installed skills list and workspace files, to confirm the VSS skills + workspace install succeeded,
- the registered `vss_orchestrator` MCP server (`nemoclaw sandbox mcp … status`).


In [ ]:
from datetime import datetime, timezone
import json
import re
import shlex
import shutil
import subprocess
import time
from pathlib import Path


def run(cmd, check=False, echo=True):
    print("$", shlex.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, check=check)
    if echo:
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
    return r


def _openshell_forward_running(port, sandbox_name):
    port = str(port)
    listing = subprocess.run(["openshell", "forward", "list"], capture_output=True, text=True)
    clean_listing = re.sub(r"\x1b\[[0-9;]*m", "", f"{listing.stdout}\n{listing.stderr}")
    for line in clean_listing.splitlines():
        parts = line.split()
        if len(parts) >= 5 and parts[0] == sandbox_name and parts[2] == port and parts[-1].lower() == "running":
            return True
    return False


def _openshell_forward_process_running(port, sandbox_name):
    port = str(port)
    markers = (
        f"openshell forward start {port} {sandbox_name}",
        f"openshell forward start --background {port} {sandbox_name}",
    )
    result = subprocess.run(["ps", "-eo", "args="], capture_output=True, text=True)
    if result.returncode != 0:
        return False
    return any(any(marker in line for marker in markers) for line in result.stdout.splitlines())


def _openshell_forward_owned_by_sandbox(port, sandbox_name):
    return _openshell_forward_running(port, sandbox_name) or _openshell_forward_process_running(port, sandbox_name)


def _dashboard_healthy(port):
    health_url = f"http://127.0.0.1:{port}/health"
    health = subprocess.run(["curl", "-fsS", health_url], capture_output=True, text=True)
    return health.returncode == 0


def ensure_dashboard_forward(port=18789):
    port = str(port)
    health_url = f"http://127.0.0.1:{port}/health"
    if _dashboard_healthy(port) and _openshell_forward_owned_by_sandbox(port, NEMOCLAW_SANDBOX_NAME):
        return
    if _dashboard_healthy(port):
        raise RuntimeError(
            f"Dashboard health is responding on {health_url}, but it is not a running OpenShell forward "
            f"for sandbox {NEMOCLAW_SANDBOX_NAME}. Stop that listener before continuing."
        )

    subprocess.run(["openshell", "forward", "stop", port, NEMOCLAW_SANDBOX_NAME], check=False, capture_output=True, text=True)
    if shutil.which("setsid"):
        subprocess.run(
            ["setsid", "-f", "openshell", "forward", "start", port, NEMOCLAW_SANDBOX_NAME],
            stdin=subprocess.DEVNULL,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        )
    else:
        subprocess.run(
            ["openshell", "forward", "start", "--background", port, NEMOCLAW_SANDBOX_NAME],
            capture_output=True,
            text=True,
            check=False,
        )

    for _ in range(10):
        if _dashboard_healthy(port) and _openshell_forward_owned_by_sandbox(port, NEMOCLAW_SANDBOX_NAME):
            return
        time.sleep(1)
    raise RuntimeError(f"Dashboard port-forward on {port} is not reachable at {health_url}")


print("Verification time (UTC):", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S %Z"))
print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("Expected policy file:", POLICY_PATH)

policy_text = Path(POLICY_PATH).read_text()
preset_name_match = re.search(r"^\s+name:\s*(\S+)", policy_text, re.MULTILINE)
print("Expected preset name:", preset_name_match.group(1) if preset_name_match else "unknown")

sandbox_result = run(["openshell", "sandbox", "get", NEMOCLAW_SANDBOX_NAME], echo=False)
sandbox_summary = "\n".join(part for part in (sandbox_result.stdout, sandbox_result.stderr) if part)
sandbox_summary = re.sub(r"\x1b\[[0-9;]*m", "", sandbox_summary)
phase_match = re.search(r"Phase:\s+(.+)", sandbox_summary)
namespace_match = re.search(r"Namespace:\s+(.+)", sandbox_summary)
sandbox_id_match = re.search(r"Id:\s+(.+)", sandbox_summary)
print("Sandbox namespace:", namespace_match.group(1).strip() if namespace_match else "unknown")
print("Sandbox phase:", phase_match.group(1).strip() if phase_match else "unknown")
print("Sandbox id:", sandbox_id_match.group(1).strip() if sandbox_id_match else "unknown")

policy_result = run(["openshell", "policy", "get", NEMOCLAW_SANDBOX_NAME])

policy_summary = "\n".join(part for part in (policy_result.stdout, policy_result.stderr) if part)
status_match = re.search(r"Status:\s+(.+)", policy_summary)
active_match = re.search(r"Active:\s+(.+)", policy_summary)
hash_match = re.search(r"Hash:\s+([0-9a-f]+)", policy_summary)
print("Active policy status:", status_match.group(1).strip() if status_match else "unknown")
print("Active policy version:", active_match.group(1).strip() if active_match else "unknown")
print("Active policy hash:", hash_match.group(1) if hash_match else "unknown")

gateway_container = resolve_openshell_gateway_container(NEMOCLAW_SANDBOX_NAME)
print("Gateway container:", gateway_container)

if not WORKSPACE_DIR.is_dir():
    raise FileNotFoundError(f"Missing plugin workspace source dir: {WORKSPACE_DIR}")
expected_workspace_md = tuple(sorted(p.name for p in WORKSPACE_DIR.glob("*.md")))
if not expected_workspace_md:
    raise RuntimeError(f"No .md files found in {WORKSPACE_DIR}; cannot verify workspace install.")
print("Expected workspace .md files (from plugin source):", list(expected_workspace_md))

if OPENCLAW_HOOKS_ENABLED:
    if not OPENCLAW_HOOKS_TOKEN:
        raise RuntimeError("OPENCLAW_HOOKS_TOKEN is required to verify OpenClaw hooks.")

    ensure_dashboard_forward(18789)
    hooks_path = "/" + OPENCLAW_HOOKS_PATH.strip("/")
    hooks_url = f"http://127.0.0.1:18789{hooks_path}/agent"
    hooks_payload = json.dumps(
        {
            "name": "NemoClaw notebook verification",
            "message": "test",
        }
    )
    hooks_cmd = [
        "curl",
        "-sS",
        "-X",
        "POST",
        hooks_url,
        "-H",
        f"Authorization: Bearer {OPENCLAW_HOOKS_TOKEN}",
        "-H",
        "Content-Type: application/json",
        "-d",
        hooks_payload,
    ]
    print("Testing OpenClaw hooks:")
    print("$", shlex.join(hooks_cmd))
    hooks_result = subprocess.run(hooks_cmd, capture_output=True, text=True)
    if hooks_result.stdout:
        print(hooks_result.stdout)
    if hooks_result.stderr:
        print(hooks_result.stderr)
    if hooks_result.returncode != 0:
        raise RuntimeError(f"OpenClaw hooks test failed with curl exit code {hooks_result.returncode}")
    hooks_response = json.loads(hooks_result.stdout)
    if not hooks_response.get("ok"):
        raise RuntimeError(f"OpenClaw hooks test returned unexpected response: {hooks_response}")
    print("OpenClaw hooks test accepted. runId:", hooks_response.get("runId"))
else:
    print("OpenClaw hooks test skipped: OPENCLAW_HOOKS_ENABLED is false.")

if gateway_container:
    def _sandbox_exec(sandbox_cmd):
        return [
            "openshell", "sandbox", "exec", "-n", NEMOCLAW_SANDBOX_NAME, "--",
            "sh", "-lc", sandbox_cmd,
        ]

    def _in_sandbox_show(label, sandbox_cmd):
        full = _sandbox_exec(sandbox_cmd)
        print(f"\n=== {label} ===")
        print("$", shlex.join(full))
        r = subprocess.run(full, capture_output=True, text=True, stdin=subprocess.DEVNULL)
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
        return r

    print("\n=== vss_orchestrator MCP status ===")
    run(["nemoclaw", "sandbox", "mcp", NEMOCLAW_SANDBOX_NAME, "status", "vss_orchestrator"])

    skills_proc = subprocess.run(
        _sandbox_exec("openclaw skills list --json"),
        capture_output=True, text=True, stdin=subprocess.DEVNULL,
    )
    workspace_dir = None
    print("\n=== openclaw skills (non-bundled) ===")
    try:
        payload = json.loads(skills_proc.stdout)
        skills = payload["skills"] if isinstance(payload, dict) else payload
        if isinstance(payload, dict):
            workspace_dir = payload.get("workspaceDir")
        non_bundled = [s for s in skills if not s.get("bundled", False)]
        if non_bundled:
            for s in non_bundled:
                print(f"- {s['name']}")
        else:
            print("No non-bundled OpenClaw skills found.")
    except (json.JSONDecodeError, TypeError) as exc:
        print(f"Could not parse skills list JSON: {exc}")
        if skills_proc.stderr:
            print(skills_proc.stderr)

    print(f"\n=== workspace .md files in {workspace_dir or '<unknown>'} ===")
    if workspace_dir:
        ls_cmd = f"ls -1 {shlex.quote(workspace_dir)} 2>/dev/null"
        ls_proc = subprocess.run(_sandbox_exec(ls_cmd), capture_output=True, text=True, stdin=subprocess.DEVNULL)
        present = {line.strip() for line in ls_proc.stdout.splitlines() if line.strip().endswith(".md")}
        for name in expected_workspace_md:
            mark = "OK  " if name in present else "MISS"
            print(f"  {mark}  {name}")
        extra = present - set(expected_workspace_md)
        if extra:
            print(f"  (also present: {sorted(extra)})")
    else:
        print("  workspaceDir not found in skills JSON payload; check skipped.")
else:
    print("Could not determine the OpenShell gateway container; plugin/skills checks were skipped.")


## 5. Prepare the host for local NIM-backed VSS profiles

If you plan to deploy local NIM-backed VSS profiles through the orchestrator tools, complete the next three pre-steps on the host first:

1. install and configure the NGC CLI,
2. authenticate Docker to `nvcr.io`, and
3. prepare the `services/agent/` Python environment.

### 5.1 Install and configure NGC CLI

This pre-step checks whether `ngc` is already installed, installs it if needed, and writes the local NGC CLI config using `NGC_CLI_API_KEY`.

In [ ]:
import os
import platform
import shutil
import subprocess


def run(cmd: str) -> str:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{result.stderr}\n{result.stdout}")
    return result.stdout.strip()


ngc_path = shutil.which("ngc")
if ngc_path:
    version = run("ngc --version 2>&1 | head -1")
    print(f"NGC CLI already installed: {version}")
else:
    arch = platform.machine()
    filename = "ngccli_arm64.zip" if arch in ("aarch64", "arm64") else "ngccli_linux.zip"
    ngc_cli_version = "4.13.0"
    url = (
        "https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/"
        f"versions/{ngc_cli_version}/files/{filename}"
    )

    print(f"Installing NGC CLI {ngc_cli_version} ({filename})...")
    run(f"cd /tmp && curl -fL --retry 3 --retry-delay 2 -o ngc_cli.zip '{url}'")

    size = os.path.getsize("/tmp/ngc_cli.zip")
    if size < 1000:
        raise RuntimeError(
            f"NGC CLI download failed: /tmp/ngc_cli.zip is only {size} bytes."
        )

    run("cd /tmp && unzip -o ngc_cli.zip")
    run("sudo cp -r /tmp/ngc-cli/* /usr/local/bin/")
    run("rm -rf /tmp/ngc_cli.zip /tmp/ngc-cli")

    version = run("ngc --version 2>&1 | head -1")
    print(f"Installed NGC CLI: {version}")


if not NGC_CLI_API_KEY:
    raise RuntimeError(
        "NGC_CLI_API_KEY is not set. Set it in the notebook settings cell or export it "
        "in the environment before running this step."
    )

print("Configuring NGC CLI...")
ngc_dir = os.path.expanduser("~/.ngc")
os.makedirs(ngc_dir, exist_ok=True)

with open(os.path.join(ngc_dir, "config"), "w") as f:
    f.write(f""";WARNING - This is a machine generated file. Do not edit manually.
;WARNING - To update local config settings, see 'ngc config set -h'.

[CURRENT]
apikey = {NGC_CLI_API_KEY}
format_type = ascii
org = nvidia
""")

print("NGC CLI configured.")
print(run("ngc config current"))

### 5.2 Docker login to `nvcr.io`

If you plan to deploy local NIM-backed VSS profiles, authenticate Docker to the NVIDIA Container Registry before using the orchestrator deployment tools.

This pre-step runs `docker login nvcr.io` with `NGC_CLI_API_KEY`.

In [ ]:
import subprocess

if not NGC_CLI_API_KEY:
    raise RuntimeError("NGC_CLI_API_KEY is not set. Export it before running this cell.")

login_result = subprocess.run(
    [
        "docker",
        "login",
        "nvcr.io",
        "--username",
        "$oauthtoken",
        "--password",
        NGC_CLI_API_KEY,
    ],
    capture_output=True,
    text=True,
)
if login_result.returncode != 0:
    raise RuntimeError(f"Docker login to nvcr.io failed\n{login_result.stderr}")

print("Docker login to nvcr.io: OK")

### 5.3 Prepare the `services/agent/` environment

The orchestrator MCP server is part of the VSS agent package, so install the Python environment in `services/agent/` before starting the server.

This runs `uv sync` from the agent directory.

In [ ]:
import os
import shutil
import subprocess

if shutil.which("uv") is None:
    raise RuntimeError("uv is not installed. Install uv first, then re-run this cell.")

uv_env = os.environ.copy()
uv_env.pop("VIRTUAL_ENV", None)
subprocess.run(["uv", "sync"], cwd=str(AGENT_DIR), env=uv_env, check=True)
venv_info = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        "import os, sys; print('uv VIRTUAL_ENV:', os.environ.get('VIRTUAL_ENV'))",
    ],
    cwd=str(AGENT_DIR),
    env=uv_env,
    check=True,
    capture_output=True,
    text=True,
)
print(venv_info.stdout.strip())
print("Environment is ready.")

## 6. Start the VSS Orchestrator MCP server and deploy a profile

From here on, the agent — not this notebook — drives the VSS deployment. Work through the sub-steps in this order (note that **6.3** is run before **6.2** in the actual UI flow):

1. **6.1** — start the host-side VSS Orchestrator MCP server (a host process listening on port `9988`).
2. **6.3** — open the OpenClaw UI, then run the smoke tests to confirm the agent can chat, sees the VSS skills, and can reach the orchestrator MCP server.
3. **6.2** — once the smoke tests pass, ask the agent in the UI to use the `vss_orchestrator__*` MCP tools to deploy and manage VSS (including teardown via `docker_down`).

**6.2** is reference material (tool list + sample prompts) and has no code cell of its own — all the work happens in the OpenClaw UI chat once **6.1** and **6.3** are done.


### 6.1 Start the VSS Orchestrator MCP server

Run the next cell to stop any previously recorded MCP server from this notebook session and start a fresh one.

In [ ]:
import importlib.util
import shutil
import signal
import subprocess
import sys
import time


helper_spec = importlib.util.spec_from_file_location("orchestrator_mcp_helper", ORCHESTRATOR_MCP_HELPER_PATH)
if helper_spec is None or helper_spec.loader is None:
    raise ImportError(f"Could not load MCP helper from {ORCHESTRATOR_MCP_HELPER_PATH}")
orchestrator_mcp_helper = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(orchestrator_mcp_helper)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

for label, ok in {
    "uv": shutil.which("uv") is not None,
    "docker": shutil.which("docker") is not None,
    "openshell": shutil.which("openshell") is not None,
    "agent dir": AGENT_DIR.is_dir(),
    "MCP config": MCP_CONFIG_PATH.is_file(),
}.items():
    if not ok:
        raise RuntimeError(f"Cannot start the MCP server because {label} is unavailable. Resolve this before proceeding.")


def _wait_for_mcp_health(process: subprocess.Popen, timeout_s: int = 60, interval_s: int = 3) -> None:
    deadline = time.time() + timeout_s
    last_error = "health check did not run"
    while time.time() < deadline:
        return_code = process.poll()
        if return_code is not None:
            raise RuntimeError(f"MCP server exited before becoming healthy with exit code {return_code}: {last_error}")
        try:
            healthy, message = orchestrator_mcp_helper.check_mcp_health(MCP_URL, AGENT_DIR)
        except subprocess.TimeoutExpired:
            healthy, message = False, "health command timed out"
        last_error = message
        if healthy:
            print(f"MCP health check passed: {message}")
            return
        time.sleep(interval_s)

    raise RuntimeError(f"MCP server did not become healthy within {timeout_s}s: {last_error}")


existing_pid = globals().get("VSS_ORCHESTRATOR_MCP_PID")
if existing_pid:
    try:
        os.kill(existing_pid, signal.SIGTERM)
        print(f"Stopped existing MCP server PID {existing_pid}")
        time.sleep(2)
    except ProcessLookupError:
        print(f"Recorded MCP server PID {existing_pid} is no longer running")

env = os.environ.copy()
env.setdefault("PYTHONUNBUFFERED", "1")
env["BREV_LINK_DOMAIN"] = BREV_LINK_DOMAIN_RESOLVED
if NGC_CLI_API_KEY:
    env["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY
if NVIDIA_API_KEY:
    env["NVIDIA_API_KEY"] = NVIDIA_API_KEY
if HARDWARE_PROFILE:
    env["HARDWARE_PROFILE"] = HARDWARE_PROFILE
env["EXTERNAL_IP"] = EXTERNAL_IP
if VSS_LLM_ENABLE_THINKING:
    env["LLM_ENABLE_THINKING"] = VSS_LLM_ENABLE_THINKING
if LLM_DEVICE_ID:
    env["LLM_DEVICE_ID"] = LLM_DEVICE_ID
if VLM_DEVICE_ID:
    env["VLM_DEVICE_ID"] = VLM_DEVICE_ID
if VSS_LLM_ENDPOINT_URL or VSS_VLM_ENDPOINT_URL:
    env["OPENAI_API_KEY"] = VSS_OPENAI_API_KEY or NVIDIA_API_KEY
if VSS_LLM_ENDPOINT_URL:
    env["LLM_ENDPOINT_URL"] = VSS_LLM_ENDPOINT_URL
if VSS_VLM_ENDPOINT_URL:
    env["VLM_ENDPOINT_URL"] = VSS_VLM_ENDPOINT_URL
log_handle = LOG_PATH.open("w")
process = subprocess.Popen(
    [
        "uv",
        "run",
        "nat",
        "mcp",
        "serve",
        "--config_file",
        str(MCP_CONFIG_PATH),
        "--host",
        MCP_HOST,
        "--port",
        str(MCP_PORT),
    ],
    cwd=str(AGENT_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
VSS_ORCHESTRATOR_MCP_PID = process.pid
print(f"Started MCP server with PID {VSS_ORCHESTRATOR_MCP_PID}")
_wait_for_mcp_health(process)
print("MCP log:", LOG_PATH)
print("MCP URL:", MCP_URL)

### 6.2 Deploy VSS from the OpenClaw UI via MCP tools

With the MCP server running, the **OpenClaw agent** can now drive the VSS deployment for you through chat. Open the **OpenClaw UI** (use **6.3** below to print a fresh link and smoke-test it), then ask the agent to use the orchestrator tools.

#### Available `vss_orchestrator__*` MCP tools

The MCP server exposes nine tools (all prefixed `vss_orchestrator__`):

| Tool | Purpose |
| --- | --- |
| `profiles` | List supported deployment profiles (`base`, `search`, `alerts`, `lvs`). |
| `prereqs` | Run Docker / GPU / NGC prerequisite checks on the host. |
| `docker_generate` | Resolve `.env` + compose YAML artifacts for the chosen profile. |
| `docker_read` | Fetch generated env/yaml by `docker_compose_id`. |
| `docker_up` | `docker compose up -d --build --quiet-pull` for the generated artifacts. |
| `docker_status` | Poll status/logs of the most recent `docker_up` / `docker_down` operation. |
| `docker_list` | List currently running container names. |
| `docker_logs` | Fetch docker logs for a given container name. |
| `docker_down` | `docker compose down -v --remove-orphans` to tear the deployment back down. |

#### Sample prompts to trigger them

You do **not** call these tools by name — the OpenClaw agent picks the right tool from your natural-language request and chains them in the correct order. Try prompts like:

| Sample prompt | Tool(s) invoked |
| --- | --- |
| *"List the available VSS deployment profiles."* | `profiles` |
| *"Check that my host meets the prerequisites for the `alerts` profile."* | `prereqs` |
| *"Deploy the VSS `alerts` profile in `verification` mode."* | `docker_generate` → `docker_up` → `docker_status` |
| *"Show me the status of the deployment."* | `docker_status` |
| *"List the running VSS containers."* | `docker_list` |
| *"Fetch the last 200 lines of logs from `vss-alert-bridge`."* | `docker_logs` |
| *"Tear down the VSS deployment."* | `docker_down` |

The agent will ask before destructive steps (e.g. `docker_down`) and stream progress back into the chat. If a tool call fails, paste the error message back to the agent and ask it to remediate — it has access to `docker_logs` and `docker_status` to diagnose.


### 6.3 Open the OpenClaw UI and verify it

Now open the **OpenClaw UI** and run a few quick checks to confirm the agent can chat, sees the VSS skills, and can reach the orchestrator MCP server you started in **6.1**. Once these checks pass, return to **6.2** and drive the deployment via the agent.

#### Step 1. Open the OpenClaw UI

Run the next cell to print a fresh **OpenClaw UI** link, then open it in your browser.

<span style="color:red"><strong>Not on Brev?</strong> If you are accessing the UI from a different machine, open an SSH tunnel before opening the OpenClaw web UI from below:<br/><code>ssh -L 18789:127.0.0.1:18789 &lt;user&gt;@&lt;nemoclaw-host&gt;</code><br/></span>

In [ ]:
import os
import subprocess


def read_etc_environment():
    env = {}
    try:
        with open("/etc/environment", encoding="utf-8") as fp:
            for raw in fp:
                line = raw.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                env[key.strip()] = value.strip().strip('"').strip("'")
    except OSError:
        return env
    return env


def fetch_gateway_token():
    result = subprocess.run(
        ["nemoclaw", NEMOCLAW_SANDBOX_NAME, "gateway-token", "--quiet"],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip()


gateway_container = resolve_openshell_gateway_container(NEMOCLAW_SANDBOX_NAME)
print("Gateway container:", gateway_container)

if not gateway_container:
    raise RuntimeError("Could not determine the OpenShell gateway container; no UI link generated.")

env_id = os.environ.get("BREV_ENV_ID", "").strip() or read_etc_environment().get("BREV_ENV_ID", "").strip()
dashboard_port = os.environ.get("NEMOCLAW_DASHBOARD_PORT", "18789").strip() or "18789"

if env_id:
    link_domain = os.environ.get("BREV_LINK_DOMAIN", "").strip() or detect_brev_link_domain()
    link_prefix = os.environ.get("BREV_LINK_PREFIX", "").strip() or str(DASHBOARD_PORT)
    origin = f"https://{link_prefix}-{env_id}.{link_domain}"
else:
    origin = f"http://127.0.0.1:{dashboard_port}"
    nemoclaw_host = subprocess.run(
        ["hostname", "-I"], capture_output=True, text=True, check=True
    ).stdout.split()[0]
    ssh_user = os.environ.get("USER", "ubuntu")
    RED, RESET = "\033[31m", "\033[0m"
    print(f"{RED}Make sure an SSH tunnel is running on your laptop before opening the OpenClaw UI:{RESET}")
    print(f"{RED}  $ ssh -L {dashboard_port}:localhost:{dashboard_port} {ssh_user}@{nemoclaw_host}{RESET}")

token = fetch_gateway_token()
openclaw_ui_url = f"{origin}/#token={token}" if token else origin
print("OpenClaw UI:", openclaw_ui_url)

#### Step 2. Verify the LLM works in chat

Start a chat with the agent and send a simple prompt such as:

- `hello`
- `what model are you using?`
- `list your available skills`

![OpenClaw UI chat screenshot](./images/OpenClawUIChat.png)

You should get a normal model response back. If the UI opens but chat fails, re-check provider setup and the active policy. 

If you reproduce the failure from a terminal and see an OpenClaw plugin-loader stack trace, make sure you used `nemoclaw <sandbox> exec -- <command>` rather than `docker exec`; `nemoclaw exec` is the documented path and sets up workspace plugins and gateway routing.

#### Step 3. Verify the VSS skills are imported

Open the **Skills** tab in the UI and confirm the **VSS skills** are present.

![OpenClaw UI skills screenshot](./images/OpenClawUISkills.png)

#### Step 4. Verify the agent sees the `vss_orchestrator` MCP tools

Run the prompts below in order. If any step fails, re-check that **6.1** completed successfully and that the MCP server is still running on port 9988.

**Prompt A — show deployment tools:**

> *"Show me the deployment tools."*

The agent should summarize the VSS deployment tools, including the `vss_orchestrator__*` tools registered by the MCP server.

![OpenClaw UI list tools screenshot](./images/OpenClawUIListTools.png)

**Prompt B — query a tool (list profiles):**

> *"List the available VSS deployment profiles."*

The agent should invoke `vss_orchestrator__profiles` and return `base`, `search`, `alerts`, `lvs`.

![OpenClaw UI search profiles screenshot](./images/OpenClawUISearch.png)

**Prompt C — invoke a deployment:**

> *"Deploy the VSS `alerts` profile in `verification` mode."*

The agent should chain `vss_orchestrator__docker_generate` → `docker_up` → `docker_status` and stream progress back into the chat. It will ask before invoking the build, and surface a `docker_compose_id` you can reference later.

![OpenClaw UI deploy VSS screenshot](./images/OpenClawUIDeploy.png)


## 7. [OPTIONAL] Verify host reachability from inside the sandbox

<span style="color:red"><strong>Important:</strong> make sure VSS is already deployed on the host before running this step.</span>

Run the next cell only after the agent has finished deploying VSS in **section 6**. It runs `nemoclaw <sandbox> connect` and feeds a probe script into that real sandbox session, which matches the OpenClaw/NemoClaw path used by skills and tools. Do **not** replace this with `docker exec`; that can use a different path and produce misleading results.

The `STATUS` column is intentionally simple:

- `REACHABLE` — an HTTP service answered on that port. `404` and non-policy `403` still count because they prove a service is listening, even if `GET /` is not a valid or authorized route.
- `NOT_REACHABLE` — the port is blocked by policy, no service is listening, DNS failed, or the request timed out. Check the `NOTE` column for the reason.


In [ ]:
import re
import subprocess

required_vars = ("NEMOCLAW_SANDBOX_NAME", "HOST_INTERNAL_ALIAS", "TEST_PORTS")
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise RuntimeError("Required variables are missing: " + ", ".join(missing_vars))

ports = " ".join(str(port) for port in TEST_PORTS)
probe = f"""
HOST_IP="${{HOST_IP:-{HOST_INTERNAL_ALIAS}}}"
PORTS="{ports}"

echo __VSS_PORT_PROBE_BEGIN__
for port in $PORTS; do
  body="$(mktemp)"
  http_code="$(curl -sS --max-time 5 -o "$body" -w '%{{http_code}}' "http://${{HOST_IP}}:${{port}}/" 2>/tmp/portcheck.err)"
  curl_exit=$?
  body_text="$(tr '[:upper:]' '[:lower:]' < "$body")"

  if printf '%s' "$body_text" | grep -Eq 'policy_denied|egress.*denied|not allowed by policy'; then
    status="NOT_REACHABLE"
    note="blocked by policy"
  elif [ "$http_code" = "502" ] || printf '%s' "$body_text" | grep -q 'upstream_unreachable'; then
    status="NOT_REACHABLE"
    note="allowed, no service listening"
  elif [ "$http_code" != "000" ]; then
    status="REACHABLE"
    note="HTTP $http_code response"
  else
    status="NOT_REACHABLE"
    case "$curl_exit" in
      7) note="connect failed" ;;
      28) note="timeout" ;;
      *) note="curl_exit=$curl_exit" ;;
    esac
  fi

  printf '%s|%s|%s|%s\n' "$port" "$status" "$http_code" "$note"
  rm -f "$body" /tmp/portcheck.err
done
echo __VSS_PORT_PROBE_END__
exit
""".strip() + "\n"

print(f"Running reachability probe through: nemoclaw {NEMOCLAW_SANDBOX_NAME} connect")
result = subprocess.run(
    ["nemoclaw", NEMOCLAW_SANDBOX_NAME, "connect"],
    input=probe,
    capture_output=True,
    text=True,
    timeout=120,
)

lines = result.stdout.splitlines()
ansi_escape = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
try:
    begin = next(i for i, line in enumerate(lines) if line.strip() == "__VSS_PORT_PROBE_BEGIN__")
    end = next(i for i, line in enumerate(lines[begin + 1 :], start=begin + 1) if line.strip() == "__VSS_PORT_PROBE_END__")
    rows = []
    for line in lines[begin + 1 : end]:
        parts = ansi_escape.sub("", line).strip().split("|", 3)
        if len(parts) == 4 and parts[0].isdigit():
            rows.append(parts)

    print(f"{'PORT':>5}  {'STATUS':<13}  {'HTTP':>4}  NOTE")
    print(f"{'----':>5}  {'------':<13}  {'----':>4}  ----")
    for port, status, http_code, note in rows:
        print(f"{port:>5}  {status:<13}  {http_code:>4}  {note}")
except StopIteration:
    print(result.stdout)

if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"nemoclaw {NEMOCLAW_SANDBOX_NAME} connect probe failed with exit code {result.returncode}"
    )
